# MobileNetV2

This standalone notebook trains **MobileNetV2**, evaluates the float model, exports dynamic-range and full INT8 TFLite models, benchmarks CPU inference, and archives outputs.

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

ARCH = "mobilenetv2"
REPO_URL = "https://github.com/hit1363/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System.git"
REPO_DIR = "/content/An-Efficient-Multi-Crop-Leaf-Disease-Detection-System"
DATASET_BASE = "/content/drive/MyDrive/leaf_data/processed"
OUTPUT_BASE = f"/content/drive/MyDrive/leaf_models/{ARCH}"
print("Configured:", ARCH)
print("Dataset:", DATASET_BASE)
print("Outputs:", OUTPUT_BASE)

## Environment and repository setup

In [ ]:
import os
import subprocess
import sys

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-r",
    os.path.join(REPO_DIR, "requirements.txt"),
], check=True)
print("Repository and dependencies are ready.")

## Configure paths and detect classes

In [ ]:
import os
import yaml

cfg_path = os.path.join(REPO_DIR, "training", "config_mobilenetv2.yaml")
with open(cfg_path, "r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

cfg["dataset"]["data_dir"] = DATASET_BASE
for split in ("train", "val", "test"):
    cfg["dataset"][f"{split}_dir"] = os.path.join(DATASET_BASE, split)
train_dir = cfg["dataset"]["train_dir"]
if not os.path.isdir(train_dir):
    raise FileNotFoundError(f"Missing training directory: {train_dir}")
class_names = sorted(
    name for name in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, name))
)
if not class_names:
    raise ValueError(f"No class folders found in {train_dir}")
cfg["model"]["num_classes"] = len(class_names)
cfg["export"]["save_dir"] = os.path.dirname(OUTPUT_BASE)
cfg["callbacks"]["csv_logger"]["filename"] = os.path.join(OUTPUT_BASE, f"training_log_{ARCH}.csv")
cfg["callbacks"]["tensorboard"]["log_dir"] = os.path.join(OUTPUT_BASE, "tensorboard")
os.makedirs(OUTPUT_BASE, exist_ok=True)
with open(cfg_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(cfg, handle, sort_keys=False)
print(f"{ARCH}: {len(class_names)} classes")
print("Config:", cfg_path)

## Train the model

In [ ]:
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "train.py"),
    "--config",
    cfg_path,
], check=True)

## Evaluate the float model

In [ ]:
from pathlib import Path

arch_dir = Path(OUTPUT_BASE)
saved_models = sorted(arch_dir.glob("saved_model_*"), key=lambda path: path.stat().st_mtime)
h5_models = sorted(arch_dir.glob("*.h5"), key=lambda path: path.stat().st_mtime)
if saved_models:
    model_path = str(saved_models[-1])
elif h5_models:
    model_path = str(h5_models[-1])
else:
    raise FileNotFoundError(f"No exported model found in {OUTPUT_BASE}")

float_report = os.path.join(OUTPUT_BASE, "evaluation_float")
os.makedirs(float_report, exist_ok=True)
subprocess.run([
    sys.executable,
    os.path.join(REPO_DIR, "training", "evaluate.py"),
    "--model", model_path,
    "--config", cfg_path,
    "--results-dir", float_report,
], check=True)
print("Evaluated:", model_path)

## Export dynamic-range and full INT8 TFLite models

In [ ]:
quant_script = os.path.join(REPO_DIR, "quantization", "post_training_quant.py")
quant_dir = os.path.join(OUTPUT_BASE, "quantized")
os.makedirs(quant_dir, exist_ok=True)

dynamic_path = os.path.join(quant_dir, f"{ARCH}_dynamic.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", dynamic_path,
    "--arch", ARCH,
], check=True)

int8_path = os.path.join(quant_dir, f"{ARCH}_int8.tflite")
subprocess.run([
    sys.executable, quant_script,
    "--model_path", model_path,
    "--output_path", int8_path,
    "--representative_data", cfg["dataset"]["train_dir"],
    "--evaluate",
    "--test_data", cfg["dataset"]["test_dir"],
    "--arch", ARCH,
], check=True)
print("Exported:", dynamic_path, int8_path)

## Benchmark and evaluate TFLite models

In [ ]:
import json
import time
import numpy as np
import tensorflow as tf

benchmark_rows = []
for tflite_path in (dynamic_path, int8_path):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    input_shape = input_details["shape"]
    if input_details["dtype"] == np.uint8:
        dummy = np.random.randint(0, 256, size=input_shape, dtype=np.uint8)
    else:
        dummy = np.random.uniform(-1.0, 1.0, size=input_shape).astype(np.float32)
    for _ in range(10):
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
    times = []
    for _ in range(50):
        start = time.perf_counter()
        interpreter.set_tensor(input_details["index"], dummy)
        interpreter.invoke()
        times.append((time.perf_counter() - start) * 1000.0)
    row = {
        "model": os.path.basename(tflite_path),
        "input_dtype": np.dtype(input_details["dtype"]).name,
        "mean_ms": float(np.mean(times)),
        "std_ms": float(np.std(times)),
        "size_mb": os.path.getsize(tflite_path) / (1024 * 1024),
    }
    benchmark_rows.append(row)
    print(json.dumps(row, indent=2))

with open(os.path.join(OUTPUT_BASE, "tflite_benchmark.json"), "w", encoding="utf-8") as handle:
    json.dump(benchmark_rows, handle, indent=2)

## Archive outputs

In [ ]:
import shutil

archive_base = os.path.join(os.path.dirname(OUTPUT_BASE), f"{ARCH}_outputs")
archive_path = shutil.make_archive(archive_base, "zip", OUTPUT_BASE)
print("Archive:", archive_path)

## Outputs

The model, class labels, logs, float evaluation report, quantized TFLite files, benchmark JSON, and ZIP archive are stored under the configured output directory.